# H22b: Fixed-rank Muon-clip under extreme anisotropy

**Counterpart script:** `experiments/Tier_4_Falsification_Stress_Tests/H22b_MUON_CLIP/run_experiment.py`

This notebook is intentionally analysis-first and behavior-aligned with the script. It imports the script module and reuses its helpers/results rather than reimplementing the training logic in notebook cells.

## Truthful scope
- The default reported experiment is **not** adaptive momentum-based clipping.
- For each target condition number `kappa`, the script estimates a single scalar `k_clip` from the **mean initial gradient effective rank** pooled across the learning-rate sweep seeds and all layers.
- That scalar `k_clip` is then held fixed for Muon-clip throughout training at the corresponding `kappa`.
- The experiment measures **initial gradient effective rank** and **anisotropy**, but it **does not directly measure any noise-energy fraction**.
- An optional adaptive mode exists in the script, but it is not the default result summarized here.

## Notebook/script identity and guiding questions

Because this notebook imports the script directly, the numerical results below come from the same implementation used by the command-line entrypoint.

Questions answered here:
1. How anisotropic are the initial gradients as `kappa` increases?
2. What fixed `k_clip` does the script actually choose from those diagnostics?
3. After optimizer-specific learning-rate sweeps, does fixed-rank Muon-clip beat Muon and/or restore SGD-relative advantage at large `kappa`?
4. Where does clipping help, hurt, or fail to change the outcome?

In [ ]:
import importlib.util
import os
import platform
import sys
import time
from dataclasses import replace
from pathlib import Path

import matplotlib
if os.environ.get('H22B_NOTEBOOK_RUN_MODE', '').strip().lower() == 'smoke':
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

plt.style.use('seaborn-v0_8-whitegrid')


def find_experiment_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'experiments' / 'Tier_4_Falsification_Stress_Tests' / 'H22b_MUON_CLIP',
        Path.cwd().parent,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if (candidate / 'run_experiment.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate H22b_MUON_CLIP/run_experiment.py. '
        'Run the notebook either from its experiment directory or from the repository root.'
    )


EXPERIMENT_DIR = find_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'
MODULE_NAME = 'h22b_muon_clip'

spec = importlib.util.spec_from_file_location(MODULE_NAME, SCRIPT_PATH)
h22b = importlib.util.module_from_spec(spec)
sys.modules[MODULE_NAME] = h22b
spec.loader.exec_module(h22b)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Script path:          {SCRIPT_PATH}')
print(f'Notebook path:        {NOTEBOOK_PATH}')

## Reproducibility and runtime notes

- Full mode runs the script's default experiment exactly: 5 `kappa` values, 5 seeds, and optimizer-specific learning-rate sweeps.
- Runtime can be on the order of minutes because Muon-clip repeatedly computes SVDs inside the learning-rate sweeps.
- For smoke-testing or quick notebook execution, set `H22B_NOTEBOOK_RUN_MODE=smoke` before execution. The notebook then uses a smaller configuration while preserving the same code path.

In [ ]:
RUN_MODE = os.environ.get('H22B_NOTEBOOK_RUN_MODE', 'full').strip().lower()
if RUN_MODE not in {'full', 'smoke'}:
    raise ValueError(f'Unsupported H22B_NOTEBOOK_RUN_MODE={RUN_MODE!r}')

DEFAULT_CONFIG = h22b.get_default_config()
if RUN_MODE == 'smoke':
    NOTEBOOK_CONFIG = replace(
        DEFAULT_CONFIG,
        kappa_values=(1, 100),
        num_steps=20,
        num_seeds=2,
        sweep_seed_count=1,
    )
else:
    NOTEBOOK_CONFIG = DEFAULT_CONFIG

CONFIG_DICT = h22b.config_to_dict(NOTEBOOK_CONFIG)
SEEDS, SWEEP_SEEDS = h22b.get_seeds(NOTEBOOK_CONFIG)

print(f'Python:      {sys.version.split()[0]}')
print(f'NumPy:       {np.__version__}')
print(f'Pandas:      {pd.__version__}')
print(f'Matplotlib:  {plt.matplotlib.__version__}')
print(f'Platform:    {platform.platform()}')
print(f'Working dir: {Path.cwd().resolve()}')
print(f'Run mode:    {RUN_MODE}')
print('Full mode may take minutes because Muon-clip repeatedly performs SVDs during LR sweeps.')
print(f'Evaluation seeds: {SEEDS}')
print(f'LR sweep seeds:   {SWEEP_SEEDS}')

pd.Series(CONFIG_DICT, name='value')

## Run the experiment through the script API

This cell executes the same experiment logic used by the script entrypoint. In full mode, the clip setting should be `fixed_init_erank`, which means one scalar `k_clip` is chosen per `kappa` from the initial gradient diagnostics and then held fixed during training.

In [ ]:
start_time = time.time()
results = h22b.run_experiment(config=NOTEBOOK_CONFIG, verbose=False)
runtime_sec = time.time() - start_time

assert results['config']['clip_mode'] == NOTEBOOK_CONFIG.clip_mode
assert Path(results['script_path']).resolve() == SCRIPT_PATH.resolve()
assert Path(results['counterpart_notebook_path']).name == 'run_experiment.ipynb'

print(f'Experiment completed in {runtime_sec:.2f} seconds.')
print(f"Reported clip mode: {results['config']['clip_mode']}")
print('Notebook/script alignment check passed: analysis is built from the script return structure.')

## Build analysis tables from the returned result structure

The script now returns enough raw information to support a serious notebook: configuration, seeds, initial diagnostics, full learning-rate sweep outcomes, chosen best learning rates, per-seed final losses, finite/divergence counts, summary ratios, and heuristic flags.

In [ ]:
summary_df = pd.DataFrame(results['summary_rows']).sort_values('kappa').reset_index(drop=True)

init_seed_rows = []
init_layer_rows = []
lr_rows = []
final_loss_rows = []
for kappa_result in results['kappa_results']:
    kappa = kappa_result['kappa']

    for row in kappa_result['init_diagnostics']['per_seed_means']:
        init_seed_rows.append({'kappa': kappa, **row})
    for row in kappa_result['init_diagnostics']['layer_records']:
        init_layer_rows.append({'kappa': kappa, **row})

    for opt, sweep in kappa_result['lr_sweeps'].items():
        for candidate in sweep['candidate_results']:
            lr_rows.append({
                'kappa': kappa,
                'optimizer': opt,
                'lr': candidate['lr'],
                'mean_finite_loss': candidate['mean_finite_loss'],
                'std_finite_loss': candidate['std_finite_loss'],
                'finite_count': candidate['finite_count'],
                'diverged_count': candidate['diverged_count'],
                'is_best': candidate['is_best'],
            })

    for opt, evaluation in kappa_result['evaluations'].items():
        for record in evaluation['loss_records']:
            final_loss_rows.append({
                'kappa': kappa,
                'optimizer': opt,
                **record,
            })

init_seed_df = pd.DataFrame(init_seed_rows)
init_layer_df = pd.DataFrame(init_layer_rows)
lr_sweep_df = pd.DataFrame(lr_rows)
final_losses_df = pd.DataFrame(final_loss_rows)

optimizer_labels = {'sgd': 'SGD', 'muon': 'Muon', 'muon_clip': 'Muon-clip'}
optimizer_colors = {'sgd': '#1f77b4', 'muon': '#2ca02c', 'muon_clip': '#d62728'}
loss_prefix = {'sgd': 'sgd', 'muon': 'muon', 'muon_clip': 'clip'}

summary_display = summary_df[[
    'kappa',
    'clip_mode',
    'k_clip',
    'mean_effective_rank',
    'std_effective_rank',
    'mean_anisotropy',
    'std_anisotropy',
    'best_sgd_lr',
    'best_muon_lr',
    'best_clip_lr',
    'adv_muon',
    'adv_clip',
    'clip_vs_muon',
]].rename(columns={
    'clip_mode': 'clip mode',
    'k_clip': 'k_clip',
    'mean_effective_rank': 'mean erank',
    'std_effective_rank': 'sd erank',
    'mean_anisotropy': 'mean anisotropy',
    'std_anisotropy': 'sd anisotropy',
    'best_sgd_lr': 'best LR (SGD)',
    'best_muon_lr': 'best LR (Muon)',
    'best_clip_lr': 'best LR (Muon-clip)',
    'adv_muon': 'SGD / Muon',
    'adv_clip': 'SGD / Muon-clip',
    'clip_vs_muon': 'Muon / Muon-clip',
})
summary_display

## Initial gradient diagnostics tied to the actual `k_clip` estimator

These are the diagnostics that determine the fixed scalar `k_clip` used by the default experiment. They summarize the **initial gradients** over the sweep seeds and all layers. The notebook does **not** reinterpret these quantities as directly measured noise-energy fractions.

In [ ]:
init_summary_df = summary_df[[
    'kappa',
    'k_clip',
    'mean_effective_rank',
    'std_effective_rank',
    'mean_anisotropy',
    'std_anisotropy',
]].copy()
display(init_summary_df)

display(init_seed_df.rename(columns={
    'mean_effective_rank': 'seed-mean erank',
    'mean_anisotropy': 'seed-mean anisotropy',
}).sort_values(['kappa', 'seed']).reset_index(drop=True))

kappas = summary_df['kappa'].to_numpy(dtype=float)
k_clip_values = pd.to_numeric(summary_df['k_clip'], errors='coerce').to_numpy(dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].errorbar(
    kappas,
    summary_df['mean_effective_rank'],
    yerr=summary_df['std_effective_rank'],
    marker='o',
    capsize=4,
)
axes[0].set_xscale('log')
axes[0].set_xlabel('target condition number $\\kappa$')
axes[0].set_ylabel('initial gradient effective rank')
axes[0].set_title('Initial effective-rank diagnostic')

axes[1].errorbar(
    kappas,
    summary_df['mean_anisotropy'],
    yerr=summary_df['std_anisotropy'],
    marker='o',
    capsize=4,
)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('target condition number $\\kappa$')
axes[1].set_ylabel('initial gradient anisotropy')
axes[1].set_title('Initial anisotropy diagnostic')

axes[2].step(kappas, k_clip_values, where='mid', linewidth=2)
axes[2].scatter(kappas, k_clip_values, s=50)
axes[2].set_xscale('log')
axes[2].set_xlabel('target condition number $\\kappa$')
axes[2].set_ylabel('fixed $k_{clip}$ used in training')
axes[2].set_title('Actual fixed clip rank selected per $\\kappa$')

plt.tight_layout()
plt.show()

## Learning-rate sweep outcomes

Each optimizer is tuned separately at each `kappa` using the script's sweep seeds. The plots below show the **mean finite final loss** obtained at each candidate learning rate, with the chosen best learning rate highlighted.

In [ ]:
best_lr_table = summary_df[[
    'kappa',
    'best_sgd_lr',
    'best_muon_lr',
    'best_clip_lr',
]].rename(columns={
    'best_sgd_lr': 'best LR (SGD)',
    'best_muon_lr': 'best LR (Muon)',
    'best_clip_lr': 'best LR (Muon-clip)',
})
display(best_lr_table)

unique_kappas = summary_df['kappa'].tolist()
ncols = 3
nrows = int(np.ceil(len(unique_kappas) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for ax, kappa in zip(axes.flat, unique_kappas):
    sub = lr_sweep_df[lr_sweep_df['kappa'] == kappa]
    for opt, label in optimizer_labels.items():
        opt_sub = sub[(sub['optimizer'] == opt) & np.isfinite(sub['mean_finite_loss'])]
        if opt_sub.empty:
            continue
        ax.plot(
            opt_sub['lr'],
            opt_sub['mean_finite_loss'],
            marker='o',
            label=label,
            color=optimizer_colors[opt],
        )
        best_sub = opt_sub[opt_sub['is_best']]
        if not best_sub.empty:
            ax.scatter(
                best_sub['lr'],
                best_sub['mean_finite_loss'],
                s=100,
                edgecolor='black',
                linewidth=0.8,
                color=optimizer_colors[opt],
                zorder=5,
            )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'$\\kappa$ = {kappa}')
    ax.set_xlabel('learning rate')
    ax.set_ylabel('mean finite final loss')

for ax in axes.flat[len(unique_kappas):]:
    ax.axis('off')

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout()
plt.show()

## Final losses, uncertainty, and divergence counts

The table reports mean and standard deviation over **finite** final losses, together with explicit finite/divergence counts. The left panel overlays seed-level finite losses on top of the optimizer means; the right panel shows divergence counts.

In [ ]:
loss_summary_table = summary_df[[
    'kappa',
    'sgd_mean',
    'sgd_std',
    'sgd_finite_count',
    'sgd_diverged_count',
    'muon_mean',
    'muon_std',
    'muon_finite_count',
    'muon_diverged_count',
    'clip_mean',
    'clip_std',
    'clip_finite_count',
    'clip_diverged_count',
]].rename(columns={
    'sgd_mean': 'SGD mean',
    'sgd_std': 'SGD sd',
    'sgd_finite_count': 'SGD finite',
    'sgd_diverged_count': 'SGD diverged',
    'muon_mean': 'Muon mean',
    'muon_std': 'Muon sd',
    'muon_finite_count': 'Muon finite',
    'muon_diverged_count': 'Muon diverged',
    'clip_mean': 'Muon-clip mean',
    'clip_std': 'Muon-clip sd',
    'clip_finite_count': 'Muon-clip finite',
    'clip_diverged_count': 'Muon-clip diverged',
})
display(loss_summary_table)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
offset_factors = {'sgd': 0.93, 'muon': 1.00, 'muon_clip': 1.07}

for opt, label in optimizer_labels.items():
    prefix = loss_prefix[opt]
    means = summary_df[f'{prefix}_mean']
    stds = summary_df[f'{prefix}_std']
    x_mean = kappas * offset_factors[opt]
    axes[0].errorbar(
        x_mean,
        means,
        yerr=stds,
        marker='o',
        capsize=4,
        label=label,
        color=optimizer_colors[opt],
    )
    finite_sub = final_losses_df[(final_losses_df['optimizer'] == opt) & (final_losses_df['finite'])]
    axes[0].scatter(
        finite_sub['kappa'].to_numpy(dtype=float) * offset_factors[opt],
        finite_sub['loss'],
        color=optimizer_colors[opt],
        alpha=0.65,
        s=35,
    )

axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('target condition number $\\kappa$')
axes[0].set_ylabel('final loss')
axes[0].set_title('Final loss: means with seed-level finite runs')
axes[0].legend()

x = np.arange(len(summary_df))
width = 0.24
axes[1].bar(x - width, summary_df['sgd_diverged_count'], width, label='SGD', color=optimizer_colors['sgd'])
axes[1].bar(x, summary_df['muon_diverged_count'], width, label='Muon', color=optimizer_colors['muon'])
axes[1].bar(x + width, summary_df['clip_diverged_count'], width, label='Muon-clip', color=optimizer_colors['muon_clip'])
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df['kappa'].tolist())
axes[1].set_xlabel('target condition number $\\kappa$')
axes[1].set_ylabel('number of diverged runs')
axes[1].set_title('Divergence counts across evaluation seeds')
axes[1].legend()

plt.tight_layout()
plt.show()

## Advantage summaries versus `kappa`

These ratios summarize the optimizer comparisons used in the script:
- `SGD / Muon`
- `SGD / Muon-clip`
- `Muon / Muon-clip`

Values above 1 favor the denominator optimizer in the corresponding ratio. The script also stores simple heuristic flags, but those flags should be interpreted as convenience summaries rather than as definitive statistical claims.

In [ ]:
advantage_table = summary_df[[
    'kappa',
    'k_clip',
    'adv_muon',
    'adv_clip',
    'clip_vs_muon',
    'clip_better_than_muon',
    'clip_restored_vs_muon_heuristic',
    'muon_status',
]].rename(columns={
    'adv_muon': 'SGD / Muon',
    'adv_clip': 'SGD / Muon-clip',
    'clip_vs_muon': 'Muon / Muon-clip',
    'clip_better_than_muon': 'clip better than Muon?',
    'clip_restored_vs_muon_heuristic': 'restored heuristic?',
    'muon_status': 'Muon status',
})
display(advantage_table)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(kappas, summary_df['adv_muon'], marker='o', linewidth=2, label='SGD / Muon')
axes[0].plot(kappas, summary_df['adv_clip'], marker='o', linewidth=2, label='SGD / Muon-clip')
axes[0].axhline(1.0, color='black', linestyle='--', linewidth=1)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('target condition number $\\kappa$')
axes[0].set_ylabel('advantage ratio')
axes[0].set_title('SGD-relative advantage')
axes[0].legend()

axes[1].plot(kappas, summary_df['clip_vs_muon'], marker='o', linewidth=2, color=optimizer_colors['muon_clip'])
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1, label='parity')
axes[1].axhline(1.1, color='gray', linestyle=':', linewidth=1, label='script heuristic threshold')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('target condition number $\\kappa$')
axes[1].set_ylabel('Muon / Muon-clip')
axes[1].set_title('Does fixed-rank Muon-clip beat Muon?')
axes[1].legend()

plt.tight_layout()
plt.show()

## Verdict and limitations

The notebook conclusion below is intentionally narrow: it describes the evidence from the **implemented fixed-`k_clip` experiment**, not from a hypothetical adaptive clipping method.

In [ ]:
def fmt_ratio(value):
    return f'{value:.2f}x' if np.isfinite(value) else 'inf'

lines = []
lines.append('Verdict for the currently implemented experiment:')
lines.append(f"- Reported clip mode: {results['config']['clip_mode']}.")
lines.append(
    '- k_clip rule: '
    + results['kappa_results'][0]['init_diagnostics']['k_clip_rule']
)

for row in results['summary_rows']:
    relation = 'beats Muon' if row['clip_better_than_muon'] else 'does not beat Muon'
    lines.append(
        f"- kappa={row['kappa']}: k_clip={row['k_clip']}, "
        f"SGD/Muon={fmt_ratio(row['adv_muon'])}, "
        f"SGD/Muon-clip={fmt_ratio(row['adv_clip'])}, "
        f"Muon/Muon-clip={fmt_ratio(row['clip_vs_muon'])}; "
        f"fixed-rank clipping {relation}; Muon status={row['muon_status']}."
    )

high_rows = [row for row in results['summary_rows'] if row['kappa'] >= 1000]
if high_rows:
    restored_count = sum(row['clip_restored_vs_muon_heuristic'] for row in high_rows)
    lines.append(
        f"- High-kappa heuristic summary: {restored_count}/{len(high_rows)} high-kappa settings satisfy the script's restoration heuristic."
    )

lines.append('- This notebook does not interpret effective rank or k_clip as a directly measured noise-energy fraction.')
lines.append('- Any adaptive momentum-based clipping claim would require a separate, explicitly labeled ablation.')

print('\n'.join(lines))

### Limitations and next useful follow-up

- The default evidence here is for `fixed_init_erank`, **not** adaptive momentum-based clipping.
- `k_clip` is estimated **once per `kappa`** from initial gradients and then held fixed during training.
- Reported uncertainty is limited by the number of seeds; in full mode the script still uses only 5 evaluation seeds.
- Divergence reporting is based on finite final-loss counts rather than on a richer stability diagnostic.
- Runtime is dominated by repeated SVDs inside Muon-clip and its learning-rate sweeps.
- The most natural next pass is a separate, clearly labeled ablation comparing `fixed_init_erank` against the optional `adaptive_momentum_erank` mode without changing the headline interpretation of the default experiment.